In [1]:
import sys, os, glob, time, socket
from loguru import logger
from tabulate import tabulate
import pandas as pd
import numpy as np
import joblib   
import random

In [2]:
ANTIGENS = [
    "Diphtheria",
    "Pertussis",
    "Polio",
    "Tetanus",
    "Rotavirus",
    "PCV",
    "Measles",
    "Mumps",
    "Rubella",
    "Hepatitis_B",
    "Hib",
    "HPV",
]

N_YEARS = 10
YEARS = np.arange(1, N_YEARS + 1)

# Reformat the capacity scenarios

In [3]:
capacity_scenario_names = [
    "base_capacity",
    "pandemic",
]
capacity_scenario_probs = {
    "base_capacity": 0.8418,
    "pandemic": 0.0275,
}

In [4]:
dfs = []
for scenario in capacity_scenario_names:
    temp = pd.read_excel(
        f"data\production_capacity_scenarios.xlsx",
        sheet_name=scenario,
    )
    temp["capacity_scenario"] = scenario
    temp["capacity_scenario_probability"] = capacity_scenario_probs[scenario]
    dfs.append(temp)

print(temp)
# Concatenate all the dataframes
capacity_scenarios = pd.concat(dfs, ignore_index=True)
# rename the columns
capacity_scenarios.rename(columns={"Manufacturer": "manufacturer"}, inplace=True)
# Convert years to columns
capacity_scenarios["capacity"] = capacity_scenarios[YEARS].values.tolist()
# Drop the years columns
capacity_scenarios.drop(YEARS, axis=1, inplace=True)

capacity_scenarios.to_csv("data\production_capacity_scenarios.csv", index=False)

       Manufacturer          1          2          3             4  \
0       AJ_Vaccines    7711003    7711003    7094122  3.886345e+06   
1          BB_NCIPD   39223956   39223956   39223956  2.196542e+07   
2    Bharat_Biotech   61029105   61029105   61029105  3.417630e+07   
3         Bilthoven   12048153   12048153   11084301  6.072269e+06   
4      Biological_E  164885690  164885690  164885690  9.233599e+07   
5    China_National   12812242   12812242   12812242  7.174856e+06   
6               GSK  226762686  226762686  226762686  1.269871e+08   
7      Haffkine_Bio   80972855   80972855   80972855  4.534480e+07   
8           LG_Chem   43702188   43702188   43493336  2.432703e+07   
9       Merck_Sharp   56991052   56991052   56991052  3.191499e+07   
10           PT_Bio   45911731   45911731   45911731  2.571057e+07   
11   Panacea_Biotec   10874165   10874165   10874165  6.089532e+06   
12           Pfizer   83916974   83916974   83916974  4.699351e+07   
13           Sanofi 

In [5]:
capacity_scenarios

,manufacturer,capacity_scenario,capacity_scenario_probability,capacity
0,AJ_Vaccines,base_capacity,0.8418,"[7711003.0, 7711003.0, 7711003.0, 7711003.0, 7..."
1,BB_NCIPD,base_capacity,0.8418,"[39223956.0, 39223956.0, 39223956.0, 39223956...."
2,Bharat_Biotech,base_capacity,0.8418,"[61029105.0, 61029105.0, 61029105.0, 61029105...."
3,Bilthoven,base_capacity,0.8418,"[12048153.0, 12048153.0, 12048153.0, 12048153...."
4,Biological_E,base_capacity,0.8418,"[164885690.0, 164885690.0, 164885690.0, 164885..."
5,China_National,base_capacity,0.8418,"[12812242.0, 12812242.0, 12812242.0, 12812242...."
6,GSK,base_capacity,0.8418,"[226762686.0, 226762686.0, 226762686.0, 226762..."
7,Haffkine_Bio,base_capacity,0.8418,"[80972855.0, 80972855.0, 80972855.0, 80972855...."
8,LG_Chem,base_capacity,0.8418,"[43702188.0, 43702188.0, 43702188.0, 43702188...."
9,Merck_Sharp,base_capacity,0.8418,"[56991052.0, 56991052.0, 56991052.0, 56991052...."


In [5]:
import pandas as pd
import numpy as np

# Load data
demand_scenarios_df = pd.read_csv(
    "data/antigen_demand_80_20_2_scenarios.csv",
    converters={"demands": pd.eval},
    usecols=["demands", "antigen", "demand_SID", "prob"],
)

capacity_scenarios_df = pd.read_csv(
    "data/production_capacity_scenarios.csv",
    converters={"capacity": pd.eval},
)

# Function to generate pairs
def generate_pairs(demand: pd.DataFrame, capacity: pd.DataFrame, n_pairs: int = 10, verbose: bool = True):
    demand_dict = demand.drop_duplicates(subset=["demand_SID"]).set_index("demand_SID")["prob"].to_dict()
    capacity_dict = capacity.drop_duplicates(subset=["capacity_scenario"]).set_index("capacity_scenario")["capacity_scenario_probability"].to_dict()

    demand_keys = list(demand_dict.keys())
    demand_probs = list(demand_dict.values())

    capacity_keys = list(capacity_dict.keys())
    capacity_probs = list(capacity_dict.values())

    if verbose:
        print(f"Unique demand scenarios: {demand_keys}")
        print(f"Unique capacity scenarios: {capacity_keys}")

    pairs = []
    pair_dfs = []
    prob_dict = {}
    pair_idx = 1

    for selected_capacity, selected_prob_capacity in capacity_dict.items():
        available_demand_keys = demand_keys.copy()
        available_demand_probs = demand_probs.copy()

        for _ in range(n_pairs):
            # Normalize the probabilities
            available_demand_probs = [p / sum(available_demand_probs) for p in available_demand_probs]

            selected_demand = np.random.choice(available_demand_keys, p=available_demand_probs)
            # Combine the probabilities
            selected_prob_demand = demand_dict[selected_demand]

            combined_prob = selected_prob_demand * selected_prob_capacity
            prob_dict[pair_idx] = combined_prob

            # Get the selected demand and capacity scenarios
            selected_demand_df = demand[demand["demand_SID"] == selected_demand].copy()
            selected_demand_df["type"] = "antigen"
            selected_demand_df.drop(columns=["prob", "demand_SID"], inplace=True)
            selected_demand_df.rename(columns={"antigen": "unit", "demands": "values"}, inplace=True)

            selected_capacity_df = capacity[capacity["capacity_scenario"] == selected_capacity].copy()
            selected_capacity_df["type"] = "manufacturer"
            selected_capacity_df.drop(columns=["capacity_scenario_probability", "capacity_scenario"], inplace=True)
            selected_capacity_df.rename(columns={"manufacturer": "unit", "capacity": "values"}, inplace=True)

            pair_df = pd.concat([selected_demand_df, selected_capacity_df])

            # Add additional information
            pair_df["pair_idx"] = pair_idx
            pair_df["pair"] = f"Demand: {selected_demand} - Capacity: {selected_capacity}"
            pair_dfs.append(pair_df)
            pairs.append((selected_demand, selected_capacity))
            pair_idx += 1

            # Remove the selected demand from the available demands
            index = available_demand_keys.index(selected_demand)
            available_demand_keys.pop(index)
            available_demand_probs.pop(index)

    pair_df = pd.concat(pair_dfs, ignore_index=True)

    # Calculate the sum of all probabilities
    total_sum = sum(prob_dict.values())

    # Scale the probabilities so they sum up to 1
    scaled_probabilities = {k: v / total_sum for k, v in prob_dict.items()}
    pair_df["pair_probability"] = pair_df["pair_idx"].map(scaled_probabilities)

    return pairs, pair_df

# Generate pairs
scenario_pairs, pair_df = generate_pairs(demand_scenarios_df, capacity_scenarios_df, n_pairs=2, verbose=True)

# Export to json and csv
pair_df.to_json("data/pair_demand_capacity_new_demand_base_capacity_pandemic.json", orient="records", lines=True)
pair_df.to_csv("data/pair_demand_capacity_new_demand_base_capacity_pandemic.csv", index=False)

# Display the DataFrame
pair_df


Unique demand scenarios: [1, 2]
Unique capacity scenarios: ['base_capacity', 'pandemic']


,unit,values,type,pair_idx,pair,pair_probability
0,Diphtheria,"[570550200, 619539900, 626455400, 594496900, 5...",antigen,1,Demand: 1 - Capacity: base_capacity,0.774692
1,HPV,"[18988200, 18779800, 27460000, 72849500, 92670...",antigen,1,Demand: 1 - Capacity: base_capacity,0.774692
2,Hepatitis_B,"[312284600, 343008200, 351717800, 334973600, 3...",antigen,1,Demand: 1 - Capacity: base_capacity,0.774692
3,Hib,"[259536200, 282777100, 287252900, 274027900, 2...",antigen,1,Demand: 1 - Capacity: base_capacity,0.774692
4,Polio,"[522182000, 582182300, 608631800, 590689600, 5...",antigen,1,Demand: 1 - Capacity: base_capacity,0.774692
...,...,...,...,...,...,...
103,PT_Bio,"[45911731.0, 45911731.0, 45911731.0, 25710569....",manufacturer,4,Demand: 2 - Capacity: pandemic,0.006327
104,Panacea_Biotec,"[10874165.0, 10874165.0, 10874165.0, 6089532.4...",manufacturer,4,Demand: 2 - Capacity: pandemic,0.006327
105,Pfizer,"[83916974.0, 83916974.0, 83916974.0, 46993505....",manufacturer,4,Demand: 2 - Capacity: pandemic,0.006327
106,Sanofi,"[87429432.0, 87429432.0, 82259017.0, 45341191....",manufacturer,4,Demand: 2 - Capacity: pandemic,0.006327


# Create scenario pairs

In [62]:
pair_df.set_index("pair_idx")

,unit,values,type,pair,pair_probability
pair_idx,,,,,
1,Diphtheria,"[570550200, 619539900, 626455400, 594496900, 5...",antigen,Demand: 1 - Capacity: base_capacity,0.8
1,HPV,"[18988200, 18779800, 27460000, 72849500, 92670...",antigen,Demand: 1 - Capacity: base_capacity,0.8
1,Hepatitis_B,"[312284600, 343008200, 351717800, 334973600, 3...",antigen,Demand: 1 - Capacity: base_capacity,0.8
1,Hib,"[259536200, 282777100, 287252900, 274027900, 2...",antigen,Demand: 1 - Capacity: base_capacity,0.8
1,Polio,"[522182000, 582182300, 608631800, 590689600, 5...",antigen,Demand: 1 - Capacity: base_capacity,0.8
1,Measles,"[420097000, 358616900, 394831600, 346093200, 3...",antigen,Demand: 1 - Capacity: base_capacity,0.8
1,Mumps,"[26052200, 26626400, 25785400, 25268100, 23536...",antigen,Demand: 1 - Capacity: base_capacity,0.8
1,PCV,"[160800200, 186996100, 209165500, 205938500, 2...",antigen,Demand: 1 - Capacity: base_capacity,0.8
1,Pertussis,"[320662700, 347414000, 352464600, 335977000, 3...",antigen,Demand: 1 - Capacity: base_capacity,0.8


In [63]:
pair_df.loc[1]

unit                                                              HPV
values              [18988200, 18779800, 27460000, 72849500, 92670...
type                                                          antigen
pair_idx                                                            1
pair                              Demand: 1 - Capacity: base_capacity
pair_probability                                                  0.8
Name: 1, dtype: object